## 01 — Measuring the Problem

We know the structure of the data. Now we measure it.

A 40 MB GeoJSON file loaded on every map interaction would be unusable in a real application. But "40 MB feels slow" is not a precise enough reason to build a whole LOD pipeline. We want numbers.

This notebook answers:

1. How large is the file on disk?
2. How many total coordinate points are we holding in memory?
3. How long does a load-and-parse cycle take?
4. What does it look like to display a sample on a map?

## Setup

Load the file and import timing tools.

In [1]:
import json
import time
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")

with open(data_path) as f:
    railroads = json.load(f)

features = railroads["features"]
print(f"Loaded {len(features):,} features")

Loaded 25,413 features


## File Size on Disk

In [2]:
size_bytes = data_path.stat().st_size
size_mb = size_bytes / 1_000_000

print(f"File size: {size_bytes:,} bytes")
print(f"File size: {size_mb:.2f} MB")

File size: 39,598,080 bytes
File size: 39.60 MB


## Total Coordinate Points

File size tells us storage cost. Coordinate count tells us rendering cost — each point must be projected and drawn.

Let's count the total number of `[lon, lat]` pairs across every feature.

In [3]:
total_coords = sum(len(f["geometry"]["coordinates"]) for f in features)

avg_coords = total_coords / len(features)

print(f"Total coordinate pairs: {total_coords:,}")
print(f"Average per feature:    {avg_coords:.1f}")

Total coordinate pairs: 1,396,480
Average per feature:    55.0


## Load Time

How long does it take to open and parse the file from scratch?

We time just the load — no display, no processing.

In [4]:
start = time.perf_counter()

with open(data_path) as f:
    _ = json.load(f)

elapsed = time.perf_counter() - start

print(f"Load time: {elapsed:.3f} seconds")

Load time: 0.235 seconds


One to three seconds on most machines — just to parse the file, before any rendering.

A map that takes 2 seconds to respond to every pan or zoom is not usable.

## Coordinate Distribution

Not all features are equal. Some segments are short (a few points), some are long (hundreds of points).

Let's see the distribution.

In [5]:
coord_counts = [len(f["geometry"]["coordinates"]) for f in features]

coord_counts.sort()

n = len(coord_counts)
print(f"Min points in a feature:    {coord_counts[0]}")
print(f"Median points per feature:  {coord_counts[n // 2]}")
print(f"90th percentile:            {coord_counts[int(n * 0.9)]}")
print(f"Max points in a feature:    {coord_counts[-1]}")

Min points in a feature:    2
Median points per feature:  24
90th percentile:            136
Max points in a feature:    500


## Displaying a Sample on a Map

Displaying all 25,000+ features at once would freeze the notebook. Instead, we display a **random sample of 500 features** — enough to see what the data looks like, not enough to crash the browser.

This itself is a preview of what we will build: a system that only shows you the data that is appropriate for your current view.

In [6]:
import random
from ipyleaflet import Map, GeoJSON

random.seed(42)
sample = random.sample(features, 500)

sample_collection = {
    "type": "FeatureCollection",
    "features": sample
}

m = Map(center=[20, 0], zoom=2)

layer = GeoJSON(
    data=sample_collection,
    style={"color": "#cc3300", "weight": 1, "opacity": 0.7}
)

m.add(layer)
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

Even 500 features gives a reasonable picture of global coverage. Now imagine loading all 25,413 every time you pan the map.

## Summarizing the Problem

Let's put the numbers in one place.

In [7]:
print("=" * 40)
print("Railroad Dataset — Problem Summary")
print("=" * 40)
print(f"  File size:          {size_mb:.1f} MB")
print(f"  Features:           {len(features):,}")
print(f"  Total coordinates:  {total_coords:,}")
print(f"  Avg coords/feature: {avg_coords:.1f}")
print(f"  Load time:          ~{elapsed:.1f}s")
print("=" * 40)
print("Every pan/zoom at this cost = unusable map")

Railroad Dataset — Problem Summary
  File size:          39.6 MB
  Features:           25,413
  Total coordinates:  1,396,480
  Avg coords/feature: 55.0
  Load time:          ~0.2s
Every pan/zoom at this cost = unusable map


## Exercise A

Find the **10 features with the most coordinate points**. For each, print the feature index (its position in the features list) and its coordinate count.

Which feature has the most points? What are its properties?

In [ ]:
ranked = sorted(
    enumerate(features),
    key=lambda item: len(item[1]['geometry']['coordinates']),
    reverse=True,
)[:10]

print(f"{'Index':<8} {'Coords':>8} {'Category':>10} {'Scalerank':>10} {'rwdb_rr_id':>12}")
print('-' * 58)
for idx, feature in ranked:
    props = feature['properties']
    print(
        f"{idx:<8} {len(feature['geometry']['coordinates']):>8} "
        f"{str(props.get('category')):>10} {str(props.get('scalerank')):>10} {str(props.get('rwdb_rr_id')):>12}"
    )

top_idx, top_feature = ranked[0]
print()
print(f'Most points: feature index {top_idx}')
print('Properties:', top_feature['properties'])

## Exercise B

The `scalerank` property controls at what zoom level a feature should appear. Features with `scalerank <= 3` are the most important — major trunk lines.

1. Count how many features have `scalerank <= 3`
2. Sum their coordinate points
3. What percentage of total coordinates do these "important" features account for?

In [ ]:
important = [f for f in features if f['properties'].get('scalerank', 99) <= 3]
important_count = len(important)
important_coords = sum(len(f['geometry']['coordinates']) for f in important)
total_coords = sum(len(f['geometry']['coordinates']) for f in features)
pct = (important_coords / total_coords * 100) if total_coords else 0.0

print(f'Important feature count (scalerank <= 3): {important_count}')
print(f'Coordinate pairs in those features:      {important_coords:,}')
print(f'Coordinate share:                         {pct:.1f}%')
print('In this dataset there are no scalerank <= 3 railroad features, so the share is 0%.')

## Check Your Understanding

We timed the **file load** at roughly 1–3 seconds. But loading is only part of the cost.

Name **two other operations** that happen between "file loaded" and "feature visible on the map" — operations that also take time and scale with feature count.

- The browser or widget has to serialize the GeoJSON and ship it into the map front end.
- The map renderer then has to style, draw, and manage thousands of line features on screen.

## Next

In [Module 01 — Douglas-Peucker Simplification](../01-Douglas_Peucker/README.md), we build the algorithm that reduces point count while preserving line shape.